In [62]:
import openai
import langchain
import pinecone
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_openai import ChatOpenAI
from pinecone import Pinecone
from pinecone import ServerlessSpec


In [63]:
from dotenv import load_dotenv
load_dotenv()

True

In [64]:
import os

In [65]:
##Lets read the documents
def read_docs(directory):
    file_loader = PyPDFDirectoryLoader(directory)
    documents = file_loader.load()
    return documents

In [66]:
doc=read_docs('documents/')
len(doc)

78

In [67]:
## Devide the docs into chunks
def chunk_data(docs, chunk_size=800, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    doc = text_splitter.split_documents(docs)
    return docs

In [68]:
documents = chunk_data(docs=doc)
len(documents)

78

In [69]:
##Check if OPENAI_API_KEY is set
print("OPENAI_API_KEY is set:", "OPENAI_API_KEY" in os.environ)
if "OPENAI_API_KEY" in os.environ:
    print("API Key found (first 10 chars):", os.environ['OPENAI_API_KEY'][:10] + "...")
else:
    print("OPENAI_API_KEY is NOT set")

OPENAI_API_KEY is set: True
API Key found (first 10 chars): sk-proj-od...


In [70]:
##Embedding Technique of OPENAI
embeddings = OpenAIEmbeddings(api_key=os.environ['OPENAI_API_KEY'])
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x13d2a48b0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x13d2a4820>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [72]:
from pinecone import ServerlessSpec

os.environ["PINECONE_API_KEY"] = "pcsk_2bwHg3_PWK83jEg965i4tAbAWComBvHu1aPFAMz6BAMnfbvshst4enKbhden7BVz1NnFtZ"
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
print(pc.list_indexes())



[{
    "name": "langchainvector",
    "metric": "cosine",
    "host": "langchainvector-7rkl7oj.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}]


In [73]:
vectorstore = PineconeVectorStore.from_documents(
    documents=doc,
    embedding=embeddings,
    index_name="langchainvector",
    pinecone_api_key="pcsk_2bwHg3_PWK83jEg965i4tAbAWComBvHu1aPFAMz6BAMnfbvshst4enKbhden7BVz1NnFtZ"
)

In [74]:
##Retrieve Result
def retrieve_query(query, k=3):
    matching_results = vectorstore.similarity_search(query, k=k)
    return matching_results


In [75]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(api_key=os.environ['OPENAI_API_KEY'], model="gpt-3.5-turbo")
retriever = vectorstore.as_retriever()

In [77]:
# Query the chain
query = """

can you point out where can i can see the information about this question : 

In some Agile teams, people are encouraged to use their skills to help the team, regardless of their role. This could mean that testers help the developers write code and developers helptesters test. What is this approach called?

"""
result = llm.invoke(retriever.invoke(query)[0].page_content + "\n\nQuestion: " + query)
print(result)

content='The information about this question can be found in the section "1.5.2 Whole Team Approach" of the Certified Tester Foundation Level v4.0.1 document. It mentions the "whole team approach" as a practice from Extreme Programming where any team member with the necessary knowledge and skills can perform any task, and everyone is responsible for quality. This approach promotes collaboration and sharing of skill sets within the team for the benefit of the project.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 790, 'total_tokens': 880, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DL19rlvQBT6W2NcsLAKyAkvzgSmuw', 'service_tier': 'default', 'finish_reason'